## 1. Setup and Authentication

In [26]:
# Authentication
from huggingface_hub import login
import os

login(os.environ.get('HF_TOKEN'))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Load Model and SAE

In [27]:
import torch
from transformer_lens import HookedTransformer
from sae_lens import SAE
import psutil
import gc

def print_gpu_utilization():
    """Print current GPU memory usage."""
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

def print_system_utilization():
    """Print current system memory usage."""
    process = psutil.Process(os.getpid())
    print(f"CPU memory used: {process.memory_info().rss / 1024**3:.2f} GB")
    print(f"System memory used: {psutil.virtual_memory().used / 1024**3:.2f} GB")
    print(f"System memory available: {psutil.virtual_memory().available / 1024**3:.2f} GB")

def clear_memory():
    """Clear memory cache."""
    gc.collect()
    torch.cuda.empty_cache()

def load_model(device="cuda", model_name="gemma-2-2b"):
    """Load the language model with memory monitoring."""
    print("Initial state:")
    print_gpu_utilization()
    print_system_utilization()
    print("-" * 50)

    clear_memory()
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

    try:
        print(f"Loading {model_name} with bfloat16...")
        model = HookedTransformer.from_pretrained(
            model_name,
            device=device,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=False
        )
    except RuntimeError as e:
        print(f"\n[Warning] bfloat16 failed: {e}")
        print("Retrying with float16...")
        clear_memory()
        model = HookedTransformer.from_pretrained(
            model_name,
            device=device,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=False
        )

    print("\nAfter loading:")
    print_gpu_utilization()
    print_system_utilization()
    print("\nModel loaded successfully!")
    
    return model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and SAE
torch.set_grad_enabled(True)
model = load_model(device = device)

# Load SAE
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_15/width_16k/canonical"
)

Initial state:
GPU memory allocated: 8.10 GB
GPU memory reserved: 8.48 GB
CPU memory used: 2.93 GB
System memory used: 40.00 GB
System memory available: 82.58 GB
--------------------------------------------------


Loading gemma-2-2b with bfloat16...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer

After loading:
GPU memory allocated: 15.90 GB
GPU memory reserved: 16.38 GB
CPU memory used: 4.53 GB
System memory used: 49.07 GB
System memory available: 73.74 GB

Model loaded successfully!


## 3. Load Dataset

In [28]:
from datasets import load_dataset

# Load and split dataset
dataset = load_dataset("Zirui22Ray/politics-dataset-demo")
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
dataset = split['train']
test_dataset = split['test']

print(f"Train set: {len(dataset)} samples")
print(f"Test set: {len(test_dataset)} samples")

Train set: 9000 samples
Test set: 1000 samples


## 4. Linear Concept Extractor

This class extracts concept vectors from SAE latent space using linear classifiers.

In [32]:
import numpy as np
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

class LinearConceptExtractor:
    """Extract concept vectors from SAE latent representations."""
    
    def __init__(self, sae, language_model, target_layer=None, device='cuda'):
        self.sae = sae
        self.language_model = language_model
        self.target_layer = target_layer if target_layer is not None else language_model.cfg.n_layers - 1
        self.device = device
        self.sae.to(device)
        self.d_sae = sae.cfg.d_sae
        
        print(f"Initialized LinearConceptExtractor for layer {self.target_layer+1}/{language_model.cfg.n_layers}")
    
    def precompute_latents(self, text_dataset, batch_size=16):
        """Precompute SAE latent representations for text dataset."""
        print(f"Precomputing latents for {len(text_dataset['text'])} samples...")
        
        all_latents = []
        all_labels = text_dataset['label']
        
        # Process in batches to reduce memory and improve speed
        for i in tqdm(range(0, len(text_dataset['text']), batch_size)):
            batch_texts = text_dataset['text'][i:i+batch_size]
            
            # Batch process each text (some texts may have different lengths)
            batch_latents = []
            for text in batch_texts:
                tokens = self.language_model.to_tokens(text)
                with torch.no_grad():
                    _, cache = self.language_model.run_with_cache(tokens)
                    token_residual = cache['resid_post', self.target_layer][0, -1, :]
                    latent = self.sae.encode(token_residual.unsqueeze(0)).squeeze(0).to(torch.float32).cpu()
                    batch_latents.append(latent)
            
            all_latents.extend(batch_latents)
        
        print(f"Precomputation completed: {len(all_latents)} latents")
        return all_latents, list(all_labels)
    
    def select_important_features(self, latents, labels, top_k=128):
        """Select most important features using ANOVA F-statistic."""
        # Convert to numpy array efficiently
        latents_np = np.stack([l.cpu().numpy() if isinstance(l, torch.Tensor) else l for l in latents])
        labels_np = np.array(labels)
        
        n_features = latents_np.shape[1]
        unique_labels = np.unique(labels_np)
        n_groups = len(unique_labels)
        
        # Vectorized F-statistic calculation
        overall_mean = np.mean(latents_np, axis=0)
        
        # Calculate between-group variance (vectorized)
        between_var = np.zeros(n_features)
        within_var = np.zeros(n_features)
        
        for label in unique_labels:
            mask = labels_np == label
            group_data = latents_np[mask]
            if len(group_data) > 0:
                group_mean = np.mean(group_data, axis=0)
                between_var += len(group_data) * (group_mean - overall_mean) ** 2
                within_var += np.sum((group_data - group_mean) ** 2, axis=0)
        
        between_var /= (n_groups - 1)
        within_var /= (len(latents_np) - n_groups)
        
        # Calculate F-statistic, avoid division by zero
        feature_scores = np.where(within_var > 1e-10, between_var / within_var, 0)
        
        selected_indices = np.argsort(feature_scores)[-top_k:]
        selected_latents = latents_np[:, selected_indices]
        
        print(f"Feature selection completed: {n_features} → {top_k}")
        return selected_latents, selected_indices
    
    def normalize_features(self, features):
        """Normalize features to zero mean and unit variance."""
        features_np = np.array([f.cpu().numpy() if isinstance(f, torch.Tensor) else f for f in features])
        mean = np.mean(features_np, axis=0)
        std = np.std(features_np, axis=0)
        std[std == 0] = 1
        return (features_np - mean) / std, mean, std
    
    def train_linear_classifier(self, latents, labels, val_size=0.2, batch_size=32, 
                                num_epochs=20, lr=1e-4, weight_decay=5e-2):
        """Train a linear classifier on latent representations."""
        print("Training linear classifier...")
        train_latents, test_latents, train_labels, test_labels = train_test_split(
            latents, labels, test_size=val_size, random_state=42, stratify=labels
        )
        
        train_dataset = TensorDataset(
            torch.tensor(train_latents, dtype=torch.float32),
            torch.tensor(train_labels, dtype=torch.long)
        )
        test_dataset = TensorDataset(
            torch.tensor(test_latents, dtype=torch.float32),
            torch.tensor(test_labels, dtype=torch.long)
        )
        
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
        
        criterion = nn.CrossEntropyLoss()
        classifier = nn.Linear(latents.shape[1], 2).to(self.device)
        optimizer = optim.Adam(classifier.parameters(), lr=lr, weight_decay=weight_decay)
        
        # Training loop
        for epoch in range(num_epochs):
            classifier.train()
            total_loss = 0
            
            for batch_latents, batch_labels in train_dataloader:
                batch_latents = batch_latents.to(self.device)
                batch_labels = batch_labels.to(self.device)
                
                outputs = classifier(batch_latents)
                loss = criterion(outputs, batch_labels)
                loss.backward()
                optimizer.zero_grad()
                outputs = classifier(batch_latents)
                loss = criterion(outputs, batch_labels)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_dataloader):.4f}")
        
        # Evaluation
        classifier.evaluators()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch_latents, batch_labels in test_dataloader:
                batch_latents = batch_latents.to(self.device)
                outputs = classifier(batch_latents)
                _, predicted = torch.max(outputs, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(batch_labels.numpy())
        
        test_acc = 100 * accuracy_score(all_labels, all_preds)
        test_f1 = 100 * f1_score(all_labels, all_preds, average='weighted')
        print("\nClassification report:")
        print(f"\nClassifier performance: Accuracy={test_acc:.2f}%, F1={test_f1:.2f}%")
        print("\nClassification report:")
        print(classification_report(all_labels, all_preds))
        
        model_info = {
            'input_dim': latents.shape[1],
            'test_accuracy': test_acc,
            'test_f1': test_f1,
            'target_layer': self.target_layer
        }
        
        return classifier, test_acc, test_f1, model_info
    
    def extract_difference_vector(self, classifier):
        """Extract difference vector between two classes."""
        weights = classifier.weight.detach().cpu().numpy()
        difference_vector = weights[1] - weights[0]
        return difference_vector / np.linalg.norm(difference_vector)
    
    def extract_concept_vectors(self, text_dataset, feature_dim=128, output_dir="concept_vectors"):
        """Complete pipeline for extracting concept vectors."""
        os.makedirs(output_dir, exist_ok=True)
        
        # Precompute latents
        original_latents, original_labels = self.precompute_latents(text_dataset)
        
        # Feature selection
        latents_np, selected_indices = self.select_important_features(
            original_latents, original_labels, top_k=feature_dim
        )
        del original_latents
        gc.collect()
        
        # Feature normalization
        normalized_latents, mean, std = self.normalize_features(latents_np)
        
        # Train classifier
        classifier, test_acc, test_f1, model_info = self.train_linear_classifier(
            normalized_latents, np.array(original_labels)
        )
        
        # Extract vectors
        difference_vector = self.extract_difference_vector(classifier)
        
        # Save results
        results = {
            'selected_indices': selected_indices,
            'reduced_dim': feature_dim,
            'original_dim': self.d_sae,
            'feature_mean': mean,
            'feature_std': std,
            'difference_vector': difference_vector,
        }
        
        np.save(os.path.join(output_dir, "difference_vector.npy"), difference_vector)
        torch.save(results, os.path.join(output_dir, "concept_vectors_full.pt"))
        torch.save(model_info, os.path.join(output_dir, "model_info.pt"))
        torch.save({
            'model_state_dict': classifier.state_dict(),
            'model_info': model_info

        }, os.path.join(output_dir, f"linear_classifier_layer_{self.target_layer}.pt"))
        
        print(f"\nResults saved to {output_dir}")

## 5. Extract Concept Vectors (Optional)

Run this to extract new concept vectors. Skip if you already have saved vectors.

In [ ]:
# Extract concept vectors
extractor = LinearConceptExtractor(
    sae=sae,
    language_model=model,
    target_layer=15,
    device=device
)

results, classifier = extractor.extract_concept_vectors(
    text_dataset=dataset,
    feature_dim=128,
    output_dir="politics_vectors_gemma2layer15"
)

Initialized LinearConceptExtractor for layer 16/26
Precomputing latents for 9000 samples...


  0%|          | 0/563 [00:00<?, ?it/s]

## 6. Load Concept Vectors and Extract Important Dimensions

In [ ]:
# Load saved results
result_dir = "politics_vectors_gemma2layer15"
results = torch.load(f"{result_dir}/concept_vectors_full.pt")

difference_vector = results['difference_vector']
selected_indices = results['selected_indices']

# Extract top 30 important dimensions
importance = np.abs(difference_vector)
top_reduced_indices = np.argsort(importance)[-30:][::-1]
important_dimensions = np.array(selected_indices)[top_reduced_indices]

print(f"Top 30 important dimensions in original SAE space:")
for i, idx in enumerate(important_dimensions[:10]):
    print(f"  {i+1}. SAE index {idx}")

# Save important dimensions
np.save(f"{result_dir}/political_important_dimensions.npy", important_dimensions)
print(f"\nSaved to {result_dir}/political_important_dimensions.npy")

Top 30 important dimensions in original SAE space:
  1. SAE index 5036
  2. SAE index 12798
  3. SAE index 15441
  4. SAE index 7556
  5. SAE index 7836
  6. SAE index 14956
  7. SAE index 7814
  8. SAE index 13295
  9. SAE index 9731
  10. SAE index 877

Saved to politics_vectors_gemma2layer15/political_important_dimensions.npy


/tmp/ipykernel_3021986/2425945416.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  results = torch.load(f"{result_dir}/concept_vectors_full.pt")


## 7. Visualize Features (Optional)

Visualize top features on Neuronpedia.

In [ ]:
from IPython.display import IFrame

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def get_dashboard_html(sae_release="gemma-2-2b", sae_id="15-gemmascope-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

# Display top 5 features
for i, idx in enumerate(important_dimensions[:5]):
    print(f"Feature {idx.item()}")
    html = get_dashboard_html(feature_idx=idx.item())
    display(IFrame(html, width=1200, height=300))

Feature 5036


Feature 12798


Feature 15441


Feature 7556


Feature 7836


## 8. Train Steering Vector (SSV)

Train a Semantic Steering Vector using the important dimensions.

In [ ]:
import copy
from datetime import datetime

def train_steering_vector(positive_statements, negative_statements, important_dims, model, sae, 
                         layer=15, lambda_dist=1.0, lambda_reg=0.01, lambda_lm=0.5,
                         learning_rate=0.01, max_iterations=100, batch_size=32):
    """
    Train a Semantic Steering Vector (SSV) in SAE latent space with language modeling loss.
    
    Args:
        positive_statements: Statements representing the target concept
        negative_statements: Statements representing the opposite concept
        important_dims: Important feature dimensions to use
        model: Language model
        sae: Sparse autoencoder
        layer: Target layer
        lambda_dist: Weight for distance loss
        lambda_reg: Weight for regularization
        lambda_lm: Weight for language modeling loss
        learning_rate: Learning rate
        max_iterations: Number of training iterations
        batch_size: Batch size
    
    Returns:
        ssv: Trained steering vector
        losses: Dictionary of loss values
    """
    print(f"Training SSV with {len(positive_statements)} positive and {len(negative_statements)} negative statements")
    print(f"Using {len(important_dims)} important dimensions")

    # Create float32 copy of SAE for numerical stability
    sae_float32 = copy.deepcopy(sae).cpu()
    for param in sae_float32.parameters():
        param.data = param.data.to(torch.float32)
    sae_float32.evaluators()
    
    # Initialize SSV and mask
    mask = np.zeros(sae_float32.cfg.d_sae, dtype=bool)
    mask[important_dims] = True
    ssv = np.zeros(sae_float32.cfg.d_sae)
    
    losses = {'total': [], 'distance': [], 'lm': [], 'reg': []}
    hook_name = f"blocks.{layer}.hook_resid_post"
    
    # Calculate centroids
    print("Calculating centroids...")
    positive_latents = []
    negative_latents = []
    
    for statements, latents_list in [(positive_statements, positive_latents), 
                                     (negative_statements, negative_latents)]:
        for statement in tqdm(statements, desc="Processing"):
            tokens = model.to_tokens(statement)
            with torch.no_grad():
                try:
                    _, cache = model.run_with_cache(tokens)
                    activation = cache[hook_name][0, -1, :]
                    latent = sae_float32.encode(activation.cpu().float().unsqueeze(0)).squeeze(0).cpu().numpy()
                    latents_list.append(latent)
                except Exception as e:
                    print(f"Error processing statement: {e}")
    
    if len(positive_latents) == 0 or len(negative_latents) == 0:
        print("Error: Could not process enough statements")
        return None, None
    
    positive_centroid = np.mean(np.array(positive_latents), axis=0)
    negative_centroid = np.mean(np.array(negative_latents), axis=0)
    
    # Initialize SSV from centroid difference
    initial_direction = positive_centroid - negative_centroid
    initial_norm = np.linalg.norm(initial_direction[mask])
    if initial_norm > 0:
        ssv[mask] = initial_direction[mask] / initial_norm
    
    print(f"Initial direction norm: {initial_norm:.4f}")
    
    # Optimization loop
    print("Starting optimization...")
    for iteration in range(max_iterations):
        # Sample batch
        pos_batch_indices = np.random.choice(len(positive_statements), batch_size, replace=True)
        neg_batch_indices = np.random.choice(len(negative_statements), batch_size, replace=True)
        
        distance_loss = 0
        lm_loss = 0
        distance_grad = np.zeros_like(ssv)
        lm_grad = np.zeros_like(ssv)
        processed_lm_samples = 0
        
        for i in range(batch_size):
            try:
                pos_tokens = model.to_tokens(positive_statements[pos_batch_indices[i]])
                neg_tokens = model.to_tokens(negative_statements[neg_batch_indices[i]])
                
                with torch.no_grad():
                    _, cache = model.run_with_cache(pos_tokens)
                    activation = cache[hook_name][0, -1, :]
                    pos_latent = sae_float32.encode(activation.cpu().float().unsqueeze(0)).squeeze(0).cpu().numpy()
                    
                    # Apply SSV
                    steered_latent = pos_latent + ssv
                    
                    # Distance loss: move toward positive centroid, away from negative
                    dist = np.sum((steered_latent - positive_centroid)**2) - \
                           0.5 * np.sum((steered_latent - negative_centroid)**2)
                    distance_loss += dist / batch_size
                    
                    # Gradient
                    distance_grad += (2 * (steered_latent - positive_centroid) - 
                                     (steered_latent - negative_centroid)) / batch_size
                
                # Language modeling loss (only for subset to reduce computation)
                if i < min(batch_size, 4):
                    try:
                        with torch.no_grad():
                            # Decode steered latent back to activation space
                            steered_latent_tensor = torch.tensor(steered_latent, dtype=torch.float32)
                            steered_act = sae_float32.decode(steered_latent_tensor.unsqueeze(0)).squeeze(0)
                            steered_act = steered_act.to(activation.device, activation.dtype)
                            
                            # Define hook to modify activation
                            def modify_activation(act, hook):
                                act[0, -1, :] = steered_act
                                return act
                            
                            # Run model with modified activation
                            modified_output = model.run_with_hooks(pos_tokens, fwd_hooks=[(hook_name, modify_activation)])
                            
                            # Calculate LM loss against negative statement
                            batch_lm_loss = 0
                            token_count = 0
                            for t in range(1, min(neg_tokens.size(1), 15)):
                                if t < modified_output.size(1):
                                    token_logits = modified_output[0, t-1, :]
                                    token_log_probs = torch.log_softmax(token_logits, dim=0)
                                    target_token_id = neg_tokens[0, t].item()
                                    if target_token_id < token_log_probs.size(0):
                                        batch_lm_loss += -token_log_probs[target_token_id].item()
                                        token_count += 1
                            
                            if token_count > 0:
                                batch_lm_loss /= token_count
                                lm_loss += batch_lm_loss / min(batch_size, 4)
                                
                                # Numerical gradient (sample subset of dimensions)
                                epsilon = 1e-4
                                for dim in important_dims[::3]:  # Sample every 3rd dimension
                                    perturbed_ssv = ssv.copy()
                                    perturbed_ssv[dim] += epsilon
                                    perturbed_latent = pos_latent + perturbed_ssv
                                    perturbed_act = sae_float32.decode(torch.tensor(perturbed_latent, dtype=torch.float32).unsqueeze(0)).squeeze(0)
                                    perturbed_act = perturbed_act.to(activation.device, activation.dtype)
                                    
                                    def perturbed_hook(act, hook):
                                        act[0, -1, :] = perturbed_act
                                        return act
                                    
                                    perturbed_output = model.run_with_hooks(pos_tokens, fwd_hooks=[(hook_name, perturbed_hook)])
                                    
                                    perturbed_lm_loss = 0
                                    p_token_count = 0
                                    for t in range(1, min(neg_tokens.size(1), 15)):
                                        if t < perturbed_output.size(1):
                                            p_token_logits = perturbed_output[0, t-1, :]
                                            p_token_log_probs = torch.log_softmax(p_token_logits, dim=0)
                                            target_token_id = neg_tokens[0, t].item()
                                            if target_token_id < p_token_log_probs.size(0):
                                                perturbed_lm_loss += -p_token_log_probs[target_token_id].item()
                                                p_token_count += 1
                                    
                                    if p_token_count > 0:
                                        perturbed_lm_loss /= p_token_count
                                        lm_grad[dim] += (perturbed_lm_loss - batch_lm_loss) / epsilon / min(batch_size, 4)
                                
                                processed_lm_samples += 1
                    except Exception as e:
                        print(f"LM loss error: {e}")
            
            except Exception as e:
                print(f"Error in batch: {e}")
        
        # Regularization
        reg_loss = lambda_reg * np.sum(np.abs(ssv[mask]))
        reg_grad = np.zeros_like(ssv)
        reg_grad[mask] = lambda_reg * np.sign(ssv[mask])
        
        # Update with combined losses
        if processed_lm_samples > 0:
            total_loss = lambda_dist * distance_loss + lambda_lm * lm_loss + reg_loss
            ssv -= learning_rate * (lambda_dist * distance_grad + lambda_lm * lm_grad + reg_grad)
        else:
            total_loss = lambda_dist * distance_loss + reg_loss
            ssv -= learning_rate * (lambda_dist * distance_grad + reg_grad)
        
        ssv[~mask] = 0
        
        losses['total'].append(total_loss)
        losses['distance'].append(distance_loss)
        losses['lm'].append(lm_loss)
        losses['reg'].append(reg_loss)
        
        if (iteration + 1) % 10 == 0:
            print(f"Iteration {iteration+1}/{max_iterations}, Loss: {total_loss:.4f} (dist: {distance_loss:.4f}, lm: {lm_loss:.4f}, reg: {reg_loss:.4f})")
    
    # Cleanup
    del sae_float32
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Training complete. Final SSV norm: {np.linalg.norm(ssv):.4f}")
    return ssv, losses


# Load important dimensions
important_dims = np.load('politics_vectors_gemma2layer15/political_important_dimensions.npy')

# Separate statements by label
right_statements = [item['text'] for item in dataset if item['label'] == 1]
left_statements = [item['text'] for item in dataset if item['label'] == 0]

print(f"Right-leaning: {len(right_statements)} statements")
print(f"Left-leaning: {len(left_statements)} statements")

# Train SSV
start_time = datetime.now()
ssv, losses = train_steering_vector(
    positive_statements=right_statements,
    negative_statements=left_statements,
    important_dims=important_dims,
    model=model,
    sae=sae,
    layer=15,
    max_iterations=100,
    batch_size=32
)

training_time = datetime.now() - start_time
print(f"\nTraining time: {training_time}")

# Save results
output_dir = "politics_vectors_gemma2layer15"
torch.save({
    'ssv': ssv,
    'losses': losses,
    'important_dims': important_dims,
    'training_time': str(training_time)
}, f"{output_dir}/steering_vector_results.pt")

print(f"Results saved to {output_dir}")

Right-leaning: 4511 statements
Left-leaning: 4489 statements
Training SSV with 4511 positive and 4489 negative statements
Using 30 important dimensions
Calculating centroids...


Processing:   0%|          | 0/4511 [00:00<?, ?it/s]

Processing:   0%|          | 0/4489 [00:00<?, ?it/s]

Initial direction norm: 18.7195
Starting optimization...
Iteration 10/100, Loss: 1990.9225
Iteration 20/100, Loss: 1304.8106
Iteration 30/100, Loss: 982.7343
Iteration 40/100, Loss: 2893.6889
Iteration 50/100, Loss: 1381.2545
Iteration 60/100, Loss: 1053.0952
Iteration 70/100, Loss: 1333.1539
Iteration 80/100, Loss: 1571.5868
Iteration 90/100, Loss: 1953.3678
Iteration 100/100, Loss: 1889.9013
Training complete. Final SSV norm: 12.3281

Training time: 0:26:56.637602
Results saved to politics_vectors_gemma2layer15


## 9. Test Steering Vector

Test the trained steering vector on test samples.

In [ ]:
def test_steering_vector(ssv, test_statements, model, sae, layer=15, 
                        scale_factors=[-6.0], max_new_tokens=120, temperature=0.7):
    """
    Test steering vector by generating text with different scale factors.
    
    Args:
        ssv: Steering vector
        test_statements: List of test prompts
        model: Language model
        sae: Sparse autoencoder
        layer: Target layer
        scale_factors: List of scaling factors to test
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
    
    Returns:
        results: Dictionary of generation results
    """
    # Create float32 SAE copy once and reuse
    sae_float32 = copy.deepcopy(sae).cpu()
    for param in sae_float32.parameters():
        param.data = param.data.to(torch.float32)
    sae_float32.evaluators()
    
    hook_name = f"blocks.{layer}.hook_resid_post"
    results = {scale: [] for scale in scale_factors}
    results['baseline'] = []
    
    # Pre-convert ssv scales to tensors for efficiency
    ssv_tensors = {scale: torch.tensor(ssv * scale, dtype=torch.float32) for scale in scale_factors}
    
    print(f"Testing on {len(test_statements)} statements with scales: {scale_factors}")
    
    # Helper function to generate text
    def generate_text(tokens, hook_fn=None):
        current_tokens = tokens.clone()
        for _ in range(max_new_tokens):
            if hook_fn:
                logits = model.run_with_hooks(current_tokens, fwd_hooks=[(hook_name, hook_fn)])
            else:
                logits = model(current_tokens)
            
            next_token = torch.multinomial(
                torch.softmax(logits[0, -1, :] / temperature, dim=0), 
                num_samples=1
            )
            current_tokens = torch.cat([current_tokens, next_token.unsqueeze(0)], dim=1)
        return model.to_string(current_tokens)
    
    for i, statement in enumerate(test_statements):
        if (i + 1) % 10 == 0:
            print(f"Processing {i+1}/{len(test_statements)}...")
        
        tokens = model.to_tokens(statement)
        
        # Baseline generation
        try:
            with torch.no_grad():
                baseline_text = generate_text(tokens)
                results['baseline'].append({
                    'original_input': statement,
                    'generated': baseline_text
                })
        except Exception as e:
            results['baseline'].append({
                'original_input': statement,
                'generated': f"Error: {e}"
            })
        
        # Steered generation for all scales
        for scale in scale_factors:
            try:
                # Create hook function with cached tensor
                def modify_activation(act, hook, scale_tensor=ssv_tensors[scale]):
                    last_token_act = act[0, -1, :].clone()
                    latent = sae_float32.encode(last_token_act.cpu().float().unsqueeze(0)).squeeze(0)
                    steered_latent = latent + scale_tensor
                    steered_act = sae_float32.decode(steered_latent.unsqueeze(0)).squeeze(0)
                    act[0, -1, :] = steered_act.to(last_token_act.device, last_token_act.dtype)
                    return act
                
                with torch.no_grad():
                    steered_text = generate_text(tokens, modify_activation)
                    results[scale].append({
                        'original_input': statement,
                        'generated': steered_text
                    })
            
            except Exception as e:
                results[scale].append({
                    'original_input': statement,
                    'generated': f"Error: {e}"
                })
    
    # Cleanup
    del sae_float32, ssv_tensors
    gc.collect()
    torch.cuda.empty_cache()
    
    return results


# Load trained SSV
ssv_results = torch.load('politics_vectors_gemma2layer15/steering_vector_results.pt')
ssv = ssv_results['ssv']

# Get left-leaning test statements
left_test = [item['text'] for item in test_dataset if item['label'] == 0]
test_samples = left_test[:200]  # Test on 200 samples

print(f"Testing on {len(test_samples)} left-leaning statements")

# Test with scale factor -6.0 (steer toward right-leaning)
test_results = test_steering_vector(
    ssv=ssv,
    test_statements=test_samples,
    model=model,
    sae=sae,
    layer=15,
    scale_factors=[-6.0],
    max_new_tokens=120
)

# Save results
torch.save(test_results, "politics_vectors_gemma2layer15/test_results.pt")
print("Test results saved!")

/tmp/ipykernel_3021986/448564389.py:104: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ssv_results = torch.load('politics_vectors_gemma2layer15/steering_vector_results.pt')


Testing on 200 left-leaning statements
Testing on 200 statements with scales: [-6.0]
Processing 10/200...
Processing 20/200...
Processing 30/200...
Processing 40/200...
Processing 50/200...
Processing 60/200...
Processing 70/200...
Processing 80/200...
Processing 90/200...
Processing 100/200...
Processing 110/200...
Processing 120/200...
Processing 130/200...
Processing 140/200...
Processing 150/200...
Processing 160/200...
Processing 170/200...
Processing 180/200...
Processing 190/200...
Processing 200/200...
Test results saved!


## 10. Evaluate Results with GPT-4 (Optional)

Evaluate the quality of steering using GPT-4 as a judge.

In [ ]:
test_results_15 = torch.load('Experiment/SAE-SSV/politics_vectors_gemma2layer15/test_results.pt')
test_results_13 = torch.load('Experiment/SAE-SSV/politics_vectors_gemma2layer13/test_results.pt')

/tmp/ipykernel_3021986/2891959716.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_results_15 = torch.load('Experiment/SAE-SSV/politics_vectors_gemma2layer15/test_re

In [ ]:
test_results_15

{-6.0: [{'original_input': 'Progressive tax policies can reduce income inequality and fund essential public services.',
   'generated': ['<bos>Progressive tax policies can reduce income inequality and fund essential public services. But our current progressive income tax system leaves millions of Americans without the tools they need to build and grow a meaningful financial future.\n\nThat’s why Families for Financial Security (FFS) is working to ensure that every household in the United States has access to financial tools and education. For more than 20 years, we’ve been working to end financial insecurity in the U.S. and building a financial future for all.\n\nWe believe that every household should have the opportunity to learn and build their own financial resilience. We also recognize that our communities are strongest when we come together to build']},
  {'original_input': 'Sexual orientation and gender identity should be protected classes under anti-discrimination laws.',
   'ge

In [ ]:
# Convert results to HuggingFace dataset format
from datasets import Dataset

test_results = torch.load('Experiment/SAE-SSV/politics_vectors_gemma2layer15/test_results.pt')

processed_data = []
for source_key, records in test_results.items():
    for record in records:
        processed_data.append({
            'source_key': str(source_key),
            'original_input': record['original_input'],
            'generated': record['generated']
        })

hf_dataset = Dataset.from_list(processed_data)

# Split by source key
split_datasets = {}
for key in ['-6.0', 'baseline']:
    split_datasets[key] = hf_dataset.filter(lambda x: x['source_key'] == key)
    print(f"Dataset '{key}': {len(split_datasets[key])} samples")

print("\nDataset prepared for evaluation")

/tmp/ipykernel_3021986/518552746.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_results = torch.load('Experiment/SAE-SSV/politics_vectors_gemma2layer15/test_result

Filter:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset '-6.0': 200 samples


Filter:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset 'baseline': 200 samples

Dataset prepared for evaluation


In [ ]:
split_datasets

{'-6.0': Dataset({
     features: ['source_key', 'original_input', 'generated'],
     num_rows: 200
 }),
 'baseline': Dataset({
     features: ['source_key', 'original_input', 'generated'],
     num_rows: 200
 })}

In [ ]:
# Evaluation with GPT-4 (requires OpenAI API key)
import json
import time
import random
import re

def evaluate_with_gpt4(steered_dataset, baseline_dataset, api_key, max_samples=200):
    """
    Evaluate steering effectiveness using GPT-4.
    
    Args:
        steered_dataset: Dataset with steered generations
        baseline_dataset: Dataset with baseline generations
        api_key: OpenAI API key
        max_samples: Maximum samples to evaluate
    
    Returns:
        evaluation_results: List of evaluation results
    """
    from openai import OpenAI
    
    client = OpenAI(api_key=api_key)
    
    def strip_original_input(generated, original):
        """Remove duplicated input from generated text."""
        if original in generated:
            idx = generated.find(original) + len(original)
            return generated[idx:].strip()
        return generated
    
    prompt_template = """
    Evaluate political steering. Rate each generation on a 1-5 scale:
    1=Strongly Left, 3=Neutral, 5=Strongly Right
    
    Original (Left-leaning): {input_text}
    Generation A: {text_a}
    Generation B: {text_b}
    
    Respond in JSON:
    {{
       "text_a_score": 1-5,
       "text_b_score": 1-5,
       "more_right_leaning": "A"/"B"/"Same",
       "explanation": "brief explanation"
    }}
    """
    
    results = []
    sample_count = min(len(steered_dataset), len(baseline_dataset), max_samples)
    
    print(f"Evaluating {sample_count} samples...")
    
    for i in range(sample_count):
        steered_item = steered_dataset[i]
        baseline_item = baseline_dataset[i]
        
        original = steered_item['original_input']
        steered = strip_original_input(steered_item['generated'], original)
        baseline = strip_original_input(baseline_item['generated'], original)
        
        # Randomize order
        if random.random() < 0.5:
            text_a, text_b = baseline, steered
            true_a, true_b = "baseline", "steered"
        else:
            text_a, text_b = steered, baseline
            true_a, true_b = "steered", "baseline"
        
        prompt = prompt_template.format(input_text=original, text_a=text_a, text_b=text_b)
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            
            result_text = response.choices[0].message.content
            
            # Parse JSON
            try:
                evaluation = json.loads(result_text)
            except:
                json_match = re.search(r'({[\s\S]*})', result_text)
                if json_match:
                    evaluation = json.loads(json_match.group(1))
                else:
                    evaluation = {"error": "Could not parse JSON"}
            
            evaluation['true_a'] = true_a
            evaluation['true_b'] = true_b
            evaluation['original_input'] = original
            
            results.append(evaluation)
            
            if (i + 1) % 10 == 0:
                print(f"Evaluated {i+1}/{sample_count}")
                time.sleep(2)  # Rate limiting
        
        except Exception as e:
            print(f"Error evaluating sample {i+1}: {e}")
            results.append({"error": str(e)})
    
    # Save results
    with open('evaluation_results.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"\nEvaluation complete! Results saved to evaluation_results.json")
    return results


# Run evaluation (uncomment and add your API key)
# results = evaluate_with_gpt4(
#     steered_dataset=split_datasets['-6.0'],
#     baseline_dataset=split_datasets['baseline'],
#     api_key="your-api-key-here",
#     max_samples=200
# )